# PySpark Learning Module: Delta Lake, Spark SQL & DataFrames

## Module Overview

This comprehensive PySpark tutorial covers:

1. **Writing DataFrames to Delta Tables** - Delta Lake fundamentals with hands-on examples
2. **Reading Data Using Spark SQL** - SQL operations in Spark
3. **Temporary Views** - Working with temp views for SQL queries
4. **CSV Writing** - Concepts and syntax (documentation)

---

**Note**: This workspace has DBFS disabled, so we'll focus on **Delta Lake** (which works perfectly) for hands-on practice, and cover CSV concepts theoretically.

**Learning Approach**: Each topic includes:
* Concepts and use cases
* Complete executable examples
* Code explanations
* Interview questions
* Best practices

In [0]:
# Import necessary libraries
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from pyspark.sql.functions import col, when, lit
from delta.tables import DeltaTable
from datetime import datetime, date

print("Creating sample employee dataset...")

# Define schema
employee_schema = StructType([
    StructField("employee_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", DoubleType(), True),
    StructField("hire_date", StringType(), True),
    StructField("city", StringType(), True)
])

# Sample employee data
employee_data = [
    (201, "Alice Johnson", "Engineering", 95000, "2020-01-15", "Seattle"),
    (202, "Bob Smith", "Marketing", 75000, "2020-02-20", "Portland"),
    (203, "Carol White", "Engineering", 98000, "2019-05-10", "Seattle"),
    (204, "David Brown", "Finance", 82000, "2021-03-12", "Boston"),
    (205, "Eve Davis", "HR", 68000, "2020-07-22", "Chicago"),
    (206, "Frank Miller", "Engineering", 102000, "2018-11-30", "Seattle"),
    (207, "Grace Lee", "Marketing", 79000, "2021-01-18", "Portland"),
    (208, "Henry Wilson", "Finance", 88000, "2019-09-05", "Boston"),
    (209, "Iris Moore", "Engineering", 93000, "2020-12-10", "Seattle"),
    (210, "Jack Taylor", "HR", 71000, "2021-06-25", "Chicago")
]

# Create DataFrame
df_employees = spark.createDataFrame(employee_data, schema=employee_schema)

print("\n=== Sample Employee Dataset ===")
df_employees.show()

print("\n=== Schema ===")
df_employees.printSchema()

print(f"\nTotal Records: {df_employees.count()}")
print("\n✓ Data ready for Delta Lake operations!")

Creating sample employee dataset...

=== Sample Employee Dataset ===
+-----------+-------------+-----------+--------+----------+--------+
|employee_id|         name| department|  salary| hire_date|    city|
+-----------+-------------+-----------+--------+----------+--------+
|        201|Alice Johnson|Engineering| 95000.0|2020-01-15| Seattle|
|        202|    Bob Smith|  Marketing| 75000.0|2020-02-20|Portland|
|        203|  Carol White|Engineering| 98000.0|2019-05-10| Seattle|
|        204|  David Brown|    Finance| 82000.0|2021-03-12|  Boston|
|        205|    Eve Davis|         HR| 68000.0|2020-07-22| Chicago|
|        206| Frank Miller|Engineering|102000.0|2018-11-30| Seattle|
|        207|    Grace Lee|  Marketing| 79000.0|2021-01-18|Portland|
|        208| Henry Wilson|    Finance| 88000.0|2019-09-05|  Boston|
|        209|   Iris Moore|Engineering| 93000.0|2020-12-10| Seattle|
|        210|  Jack Taylor|         HR| 71000.0|2021-06-25| Chicago|
+-----------+-------------+-------

# 1. Writing DataFrames to Delta Tables

## What is Delta Lake?

Delta Lake is an **open-source storage layer** that brings **ACID transactions**, **schema enforcement**, and **time travel** to data lakes. Built on Parquet with a transaction log.

## Why Delta Lake?

### Parquet Limitations:
* No ACID transactions - concurrent write issues
* No schema enforcement - bad data can corrupt tables
* Can't UPDATE/DELETE rows - must rewrite entire partitions
* No version history - can't rollback mistakes

### Delta Lake Advantages:
* **ACID Transactions**: All-or-nothing writes, safe concurrent access
* **Time Travel**: Query any historical version
* **Schema Enforcement**: Prevent bad data at write time
* **UPSERT/DELETE/MERGE**: Modify data efficiently
* **Audit History**: Track all changes
* **OPTIMIZE & VACUUM**: Maintain performance

## Real-World Use Cases

1. **Data Lakehouse**: Combines data lake flexibility with database reliability
2. **Streaming + Batch**: Single table for both workloads
3. **Incremental ETL**: MERGE for updates instead of full rewrites
4. **ML Feature Stores**: Versioned, reproducible features
5. **Compliance & Auditing**: Time travel for regulatory requirements

## Key Operations

```python
# Write
df.write.format("delta").mode("overwrite").saveAsTable("table_name")

# Read  
df = spark.read.table("table_name")

# Time Travel
df = spark.read.option("versionAsOf", 0).table("table_name")

# History
DeltaTable.forName(spark, "table_name").history().show()
```

In [0]:
# Example 1: Writing and Reading Delta Tables

print("="*70)
print("EXAMPLE 1: BASIC DELTA LAKE OPERATIONS")
print("="*70)

# Write DataFrame as Delta table
print("\n1A. Writing Delta Table:")
table_name = "employees_delta"

df_employees.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"   ✓ Delta table '{table_name}' created successfully")

# Read Delta table
print("\n1B. Reading Delta Table:")
df_read = spark.read.table(table_name)
print(f"   Records: {df_read.count()}")
df_read.show(5)

# View history
print("\n1C. Delta History (Transaction Log):")
delta_table = DeltaTable.forName(spark, table_name)
history = delta_table.history()
history.select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

print("\n=== Key Points ===")
print("• Delta tables are stored in Unity Catalog")
print("• Every operation is logged in _delta_log/")
print("• ACID guarantees: atomic, consistent, isolated, durable")
print("• Schema is enforced on every write")
print("\n✓ Basic Delta operations complete!")

EXAMPLE 1: BASIC DELTA LAKE OPERATIONS

1A. Writing Delta Table:
   ✓ Delta table 'employees_delta' created successfully

1B. Reading Delta Table:
   Records: 10
+-----------+-------------+-----------+-------+----------+--------+
|employee_id|         name| department| salary| hire_date|    city|
+-----------+-------------+-----------+-------+----------+--------+
|        201|Alice Johnson|Engineering|95000.0|2020-01-15| Seattle|
|        202|    Bob Smith|  Marketing|75000.0|2020-02-20|Portland|
|        203|  Carol White|Engineering|98000.0|2019-05-10| Seattle|
|        204|  David Brown|    Finance|82000.0|2021-03-12|  Boston|
|        205|    Eve Davis|         HR|68000.0|2020-07-22| Chicago|
+-----------+-------------+-----------+-------+----------+--------+
only showing top 5 rows

1C. Delta History (Transaction Log):
+-------+-------------------+---------------------------------+-----------------------------------------------------------------------------------------------------

In [0]:
# Example 2: Delta Save Modes

print("="*70)
print("EXAMPLE 2: DELTA SAVE MODES")
print("="*70)

# MODE 1: APPEND - Add new records
print("\n2A. APPEND Mode:")
print("   Add new data without removing existing records")

new_employees = [
    (211, "Kevin Brown", "Engineering", 96000, "2022-01-10", "Seattle"),
    (212, "Laura Green", "Marketing", 77000, "2022-02-15", "Portland"),
    (213, "Mike Chen", "Finance", 85000, "2022-03-20", "Boston")
]

df_new = spark.createDataFrame(new_employees, schema=employee_schema)

print("\n   New records to append:")
df_new.show()

df_new.write.format("delta").mode("append").saveAsTable("employees_delta")

count_after_append = spark.read.table("employees_delta").count()
print(f"   ✓ Appended 3 records. Total now: {count_after_append}")

# MODE 2: OVERWRITE - Replace all data
print("\n2B. OVERWRITE Mode:")
print("   Replace entire table with new data")

# Use only first 5 records
df_small = df_employees.limit(5)
df_small.write.format("delta").mode("overwrite").saveAsTable("employees_delta")

count_after_overwrite = spark.read.table("employees_delta").count()
print(f"   ✓ Overwritten. Total now: {count_after_overwrite}")

print("\n=== Save Mode Summary ===")
print("• append: Adds records (incremental loads)")
print("• overwrite: Replaces all data (full refresh)")
print("• errorIfExists: Fails if table exists (default)")
print("• ignore: Skip if table exists")
print("\n✓ Save modes demonstrated!")

EXAMPLE 2: DELTA SAVE MODES

2A. APPEND Mode:
   Add new data without removing existing records

   New records to append:
+-----------+-----------+-----------+-------+----------+--------+
|employee_id|       name| department| salary| hire_date|    city|
+-----------+-----------+-----------+-------+----------+--------+
|        211|Kevin Brown|Engineering|96000.0|2022-01-10| Seattle|
|        212|Laura Green|  Marketing|77000.0|2022-02-15|Portland|
|        213|  Mike Chen|    Finance|85000.0|2022-03-20|  Boston|
+-----------+-----------+-----------+-------+----------+--------+

   ✓ Appended 3 records. Total now: 13

2B. OVERWRITE Mode:
   Replace entire table with new data
   ✓ Overwritten. Total now: 5

=== Save Mode Summary ===
• append: Adds records (incremental loads)
• overwrite: Replaces all data (full refresh)
• errorIfExists: Fails if table exists (default)
• ignore: Skip if table exists

✓ Save modes demonstrated!


In [0]:
# Example 3: Time Travel with Delta Lake

print("="*70)
print("EXAMPLE 3: TIME TRAVEL")
print("="*70)

# Show current state
print("\n3A. Current Table (latest version):")
current = spark.read.table("employees_delta")
print(f"   Current records: {current.count()}")
current.show(3)

# Query specific version (version 0 = initial write)
print("\n3B. Time Travel to Version 0:")
version_0 = spark.read \
    .option("versionAsOf", 0) \
    .table("employees_delta")

print(f"   Version 0 had: {version_0.count()} records")
version_0.show(3)

# Show all versions
print("\n3C. View All Versions:")
delta_table = DeltaTable.forName(spark, "employees_delta")
history = delta_table.history()
history.select("version", "timestamp", "operation").show(truncate=False)

print("\n=== Time Travel Benefits ===")
print("• Recover from mistakes - query any previous version")
print("• Audit trail - see exactly what changed and when")
print("• Reproducibility - recreate analysis from specific point in time")
print("• A/B testing - compare model results across versions")
print("\n✓ Time travel demonstrated!")

EXAMPLE 3: TIME TRAVEL

3A. Current Table (latest version):
   Current records: 5
+-----------+-------------+-----------+-------+----------+--------+
|employee_id|         name| department| salary| hire_date|    city|
+-----------+-------------+-----------+-------+----------+--------+
|        201|Alice Johnson|Engineering|95000.0|2020-01-15| Seattle|
|        202|    Bob Smith|  Marketing|75000.0|2020-02-20|Portland|
|        203|  Carol White|Engineering|98000.0|2019-05-10| Seattle|
+-----------+-------------+-----------+-------+----------+--------+
only showing top 3 rows

3B. Time Travel to Version 0:
   Version 0 had: 10 records
+-----------+-------------+-----------+------+----------+--------+
|employee_id|         name| department|salary| hire_date|    city|
+-----------+-------------+-----------+------+----------+--------+
|        201|Alice Johnson|Engineering| 95000|2020-01-15| Seattle|
|        202|    Bob Smith|  Marketing| 75000|2020-02-20|Portland|
|        203|  Carol Wh

# 2. Reading Data Using Spark SQL

Spark SQL allows you to query DataFrames using familiar SQL syntax.

## Key Concepts

* **spark.sql()**: Execute SQL queries against tables and views
* **Same execution plan**: SQL and DataFrame API compile to identical execution
* **All standard SQL**: SELECT, WHERE, JOIN, GROUP BY, window functions, CTEs, etc.
* **Query optimization**: Catalyst optimizer works on both APIs

## When to Use SQL vs DataFrame API

| SQL API | DataFrame API |
|---------|---------------|
| Familiar syntax for SQL users | Type-safe, compile-time errors |
| Great for analysts | Better IDE autocomplete |
| Easy to read/share | Programmatic transformations |
| Good for complex JOINs | Better for dynamic logic |

**Best Practice**: Use both! They work together seamlessly.

In [0]:
# Example 4: Spark SQL Complete Examples

print("="*70)
print("EXAMPLE 4: SPARK SQL")
print("="*70)

# Ensure we have data
df_employees.write.format("delta").mode("overwrite").saveAsTable("employees_delta")

# 1. Basic SELECT with WHERE
print("\n4A. SELECT with WHERE:")
engineering = spark.sql("""
    SELECT * FROM employees_delta 
    WHERE department = 'Engineering'
    ORDER BY salary DESC
""")

print(f"   Engineering employees: {engineering.count()}")
engineering.show()

# 2. Aggregations with GROUP BY
print("\n4B. GROUP BY Aggregations:")
agg_query = spark.sql("""
    SELECT 
        department,
        COUNT(*) as employee_count,
        ROUND(AVG(salary), 2) as avg_salary,
        MAX(salary) as max_salary
    FROM employees_delta
    GROUP BY department
    ORDER BY avg_salary DESC
""")
agg_query.show()

# 3. CASE WHEN for conditional logic
print("\n4C. CASE WHEN (Conditional Logic):")
levels = spark.sql("""
    SELECT 
        name,
        department,
        salary,
        CASE 
            WHEN salary >= 95000 THEN 'Senior'
            WHEN salary >= 80000 THEN 'Mid'
            ELSE 'Junior'
        END as level
    FROM employees_delta
    ORDER BY salary DESC
""")
levels.show()

# 4. Common Table Expression (CTE)
print("\n4D. CTE (WITH Clause):")
cte_query = spark.sql("""
    WITH dept_stats AS (
        SELECT 
            department,
            AVG(salary) as dept_avg
        FROM employees_delta
        GROUP BY department
    )
    SELECT 
        e.name,
        e.department,
        e.salary,
        ROUND(d.dept_avg, 2) as department_avg,
        ROUND(e.salary - d.dept_avg, 2) as diff_from_avg
    FROM employees_delta e
    JOIN dept_stats d ON e.department = d.department
    ORDER BY e.department, e.salary DESC
""")
cte_query.show()

print("\n=== SQL Features Available ===")
print("• All standard SQL: SELECT, WHERE, JOIN, GROUP BY, HAVING")
print("• Window functions: ROW_NUMBER(), RANK(), LAG(), LEAD()")
print("• CTEs (WITH clause) for complex queries")
print("• Subqueries in FROM and WHERE clauses")
print("• UNION, INTERSECT, EXCEPT set operations")
print("\n✓ Spark SQL examples complete!")

EXAMPLE 4: SPARK SQL

4A. SELECT with WHERE:
   Engineering employees: 4
+-----------+-------------+-----------+--------+----------+-------+
|employee_id|         name| department|  salary| hire_date|   city|
+-----------+-------------+-----------+--------+----------+-------+
|        206| Frank Miller|Engineering|102000.0|2018-11-30|Seattle|
|        203|  Carol White|Engineering| 98000.0|2019-05-10|Seattle|
|        201|Alice Johnson|Engineering| 95000.0|2020-01-15|Seattle|
|        209|   Iris Moore|Engineering| 93000.0|2020-12-10|Seattle|
+-----------+-------------+-----------+--------+----------+-------+


4B. GROUP BY Aggregations:
+-----------+--------------+----------+----------+
| department|employee_count|avg_salary|max_salary|
+-----------+--------------+----------+----------+
|Engineering|             4|   97000.0|  102000.0|
|    Finance|             2|   85000.0|   88000.0|
|  Marketing|             2|   77000.0|   79000.0|
|         HR|             2|   69500.0|   71000.

# 3. Temporary Views

Temporary views allow you to query DataFrames using SQL without persisting them as tables.

## Types of Views

### 1. Temporary View
```python
df.createTempView("view_name")
```
* Session-scoped (exists only in current SparkSession)
* Fails if view already exists
* Automatically dropped when session ends
* Access: `SELECT * FROM view_name`

### 2. Replace Temporary View
```python
df.createOrReplaceTempView("view_name")
```
* Session-scoped
* Overwrites existing view if present
* Most commonly used

### 3. Global Temporary View
```python
df.createGlobalTempView("view_name")
```
* Cross-session (shared across SparkSessions)
* Access: `SELECT * FROM global_temp.view_name`
* Not supported on serverless compute

## When to Use Temporary Views

* **Quick SQL queries** on DataFrames without saving to catalog
* **Complex transformations** easier to express in SQL
* **Share data** between SQL and DataFrame API in same notebook
* **Testing** before creating permanent tables

In [0]:
# Example 5: Working with Temporary Views

print("="*70)
print("EXAMPLE 5: TEMPORARY VIEWS")
print("="*70)

# Create sample product data
product_data = [
    (1, "Laptop", 1200, "Electronics", 50),
    (2, "Mouse", 25, "Electronics", 200),
    (3, "Keyboard", 75, "Electronics", 150),
    (4, "Monitor", 300, "Electronics", 80),
    (5, "Desk Chair", 250, "Furniture", 40),
    (6, "Desk", 400, "Furniture", 25),
    (7, "Notebook", 5, "Office Supplies", 500),
    (8, "Pen Set", 15, "Office Supplies", 300)
]

df_products = spark.createDataFrame(
    product_data,
    ["id", "name", "price", "category", "stock"]
)

print("\nOriginal Product DataFrame:")
df_products.show()

# 1. Create Temporary View
print("\n5A. createTempView():")
df_products.createOrReplaceTempView("products")
print("   ✓ Created temporary view 'products'")

# Query the view
print("\n   Query: Products over $100")
expensive = spark.sql("""
    SELECT * FROM products 
    WHERE price > 100
    ORDER BY price DESC
""")
expensive.show()

# 2. Create or Replace Temporary View  
print("\n5B. createOrReplaceTempView():")
print("   Filtering only Electronics and replacing view")

df_products.filter("category = 'Electronics'").createOrReplaceTempView("products")
print("   ✓ Replaced 'products' view with filtered data")

print("\n   View now contains only Electronics:")
spark.sql("SELECT * FROM products").show()

# 3. Use view in complex SQL
print("\n5C. Complex Query on Temporary View:")

# Restore full product view
df_products.createOrReplaceTempView("products")

summary = spark.sql("""
    SELECT 
        category,
        COUNT(*) as product_count,
        ROUND(AVG(price), 2) as avg_price,
        SUM(stock) as total_stock,
        SUM(price * stock) as inventory_value
    FROM products
    GROUP BY category
    ORDER BY inventory_value DESC
""")

print("   Category Summary:")
summary.show()

print("\n=== Temporary View Benefits ===")
print("• No permanent storage - perfect for ephemeral data")
print("• SQL-friendly interface to DataFrames")
print("• Easy to create, replace, and query")
print("• Automatically cleaned up when session ends")
print("\n✓ Temporary views demonstrated!")

EXAMPLE 5: TEMPORARY VIEWS

Original Product DataFrame:
+---+----------+-----+---------------+-----+
| id|      name|price|       category|stock|
+---+----------+-----+---------------+-----+
|  1|    Laptop| 1200|    Electronics|   50|
|  2|     Mouse|   25|    Electronics|  200|
|  3|  Keyboard|   75|    Electronics|  150|
|  4|   Monitor|  300|    Electronics|   80|
|  5|Desk Chair|  250|      Furniture|   40|
|  6|      Desk|  400|      Furniture|   25|
|  7|  Notebook|    5|Office Supplies|  500|
|  8|   Pen Set|   15|Office Supplies|  300|
+---+----------+-----+---------------+-----+


5A. createTempView():
   ✓ Created temporary view 'products'

   Query: Products over $100
+---+----------+-----+-----------+-----+
| id|      name|price|   category|stock|
+---+----------+-----+-----------+-----+
|  1|    Laptop| 1200|Electronics|   50|
|  6|      Desk|  400|  Furniture|   25|
|  4|   Monitor|  300|Electronics|   80|
|  5|Desk Chair|  250|  Furniture|   40|
+---+----------+-----+--

# 4. Writing DataFrames to CSV (Concepts)

## Important Note

**This workspace has DBFS disabled**, which means CSV writes to local filesystem paths don't work on serverless compute. However, the concepts and syntax are essential for PySpark interviews and production environments.

## CSV Writing Syntax

### Basic Write
```python
df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/path/to/output")
```

### Key Options

| Option | Description | Example |
|--------|-------------|----------|
| `header` | Include column names | `.option("header", "true")` |
| `sep` | Field delimiter | `.option("sep", "\|")`  (pipe) |
| `quote` | Quote character | `.option("quote", '"')` |
| `nullValue` | NULL representation | `.option("nullValue", "NA")` |
| `compression` | Compress output | `.option("compression", "gzip")` |
| `dateFormat` | Date format | `.option("dateFormat", "yyyy-MM-dd")` |

### Save Modes

```python
# Overwrite - replace all data
df.write.mode("overwrite").csv(path)

# Append - add to existing
df.write.mode("append").csv(path)

# Error if exists (default)
df.write.mode("error").csv(path)

# Ignore if exists
df.write.mode("ignore").csv(path)
```

### Partitioning & File Control

```python
# Partition by column
df.write.partitionBy("department").csv(path)

# Single output file (not recommended for large data)
df.coalesce(1).write.csv(path)

# Control number of files
df.repartition(5).write.csv(path)
```

### Compression Options

```python
# GZIP - good compression, not splittable
df.write.option("compression", "gzip").csv(path)

# BZIP2 - best compression, splittable, slower
df.write.option("compression", "bzip2").csv(path)

# Snappy - fast, moderate compression
df.write.option("compression", "snappy").csv(path)
```

## Interview Questions

**Q: coalesce(1) vs repartition(1)?**  
A: `coalesce` minimizes shuffle (reduce-only), `repartition` does full shuffle (can increase or decrease)

**Q: How to write single CSV with custom name?**  
A: PySpark writes directories, not single files. Use `coalesce(1)` then rename the part file

**Q: Data types in CSV?**  
A: All types converted to strings. Must use `inferSchema` or specify schema when reading

**Q: What is _SUCCESS file?**  
A: Empty marker file indicating write completed successfully

**Q: Optimize CSV write performance?**  
A: Use compression, repartition appropriately, avoid `coalesce(1)` on large data

## Where CSV Writes Work

* **Cloud storage**: `s3://bucket/path`, `abfss://container/path`, `gs://bucket/path`
* **Unity Catalog Volumes**: `/Volumes/catalog/schema/volume/file.csv`
* **Workspaces with DBFS enabled**: `/dbfs/` paths

## Modern Alternative: Delta Lake

For most use cases, **Delta Lake is superior to CSV**:
* ACID transactions
* Schema enforcement  
* Better compression
* Faster reads/writes
* Supports UPDATE/DELETE/MERGE
* Time travel capabilities

**Recommendation**: Use Delta Lake for storage, export to CSV only when required by external systems.

# Module Summary & Interview Prep

## What We Covered

### ✓ Executable Examples (Pure PySpark)

1. **Delta Lake**
   * Write DataFrames as Delta tables
   * Save modes (append, overwrite)
   * Time travel and version history
   * Transaction log and ACID properties

2. **Spark SQL**
   * SELECT, WHERE, GROUP BY, aggregations
   * CASE WHEN conditional logic
   * CTEs (Common Table Expressions)
   * Joins and complex queries

3. **Temporary Views**
   * createTempView() vs createOrReplaceTempView()
   * SQL queries on DataFrames
   * Session-scoped views

### ✓ Conceptual Coverage

4. **CSV Writing**
   * Syntax and options
   * Save modes and partitioning
   * Compression strategies
   * Best practices

---

## Quick Reference

### Delta Lake
```python
# Write
df.write.format("delta").mode("overwrite").saveAsTable("table")

# Read
df = spark.read.table("table")

# Time Travel
df = spark.read.option("versionAsOf", 0).table("table")

# History
DeltaTable.forName(spark, "table").history().show()
```

### Spark SQL
```python
# Execute SQL
result = spark.sql("SELECT * FROM table WHERE condition")

# CTE
result = spark.sql("""
    WITH temp AS (SELECT ...)
    SELECT * FROM temp
""")
```

### Temporary Views
```python
# Create view
df.createOrReplaceTempView("view_name")

# Query view
result = spark.sql("SELECT * FROM view_name")
```

---

## Key Interview Questions

### Delta Lake

**Q: Why Delta over Parquet?**  
A: ACID transactions, schema enforcement, time travel, UPDATE/DELETE/MERGE support

**Q: What is _delta_log?**  
A: Transaction log storing all operations, enabling versioning and time travel

**Q: Explain Time Travel**  
A: Query historical versions using `versionAsOf` or `timestampAsOf` options

**Q: OPTIMIZE vs VACUUM?**  
A:
* OPTIMIZE: Compacts small files into larger ones (improves read performance)
* VACUUM: Removes old data files beyond retention period (saves storage)

### Spark SQL

**Q: SQL vs DataFrame API?**  
A: Same execution plan via Catalyst optimizer. SQL for readability, DataFrame API for type-safety

**Q: What is a CTE?**  
A: Common Table Expression (WITH clause) - named temporary result set for complex queries

**Q: Can you mix SQL and DataFrame API?**  
A: Yes! Create temp views from DataFrames, or convert SQL results back to DataFrames

### Temporary Views

**Q: createTempView vs createOrReplaceTempView?**  
A: createTempView fails if exists, createOrReplaceTempView overwrites

**Q: When do temp views expire?**  
A: When the SparkSession ends

**Q: Global temp view vs temp view?**  
A: Global temp views are cross-session (shared), regular temp views are session-scoped

---

## Best Practices

### Delta Lake
* ✓ Use Unity Catalog for governance
* ✓ Run OPTIMIZE regularly on high-write tables
* ✓ Set appropriate VACUUM retention (default 7 days)
* ✓ Enable Auto Optimize for streaming tables
* ✓ Use schema evolution carefully (mergeSchema, overwriteSchema)

### Spark SQL
* ✓ Use temp views for complex multi-step transformations
* ✓ Leverage CTEs for readability
* ✓ Avoid SELECT * in production
* ✓ Use EXPLAIN to understand query plans
* ✓ Cache only when data is reused multiple times

### General
* ✓ Always check data quality with `.show()` and `.count()`
* ✓ Use explicit schemas instead of inferSchema in production
* ✓ Test with `.limit()` on large datasets
* ✓ Monitor execution with `.explain()` and Spark UI

---

## Next Steps

1. **Practice**: Run all examples multiple times with different data
2. **Explore**: Try MERGE, UPDATE, DELETE operations on Delta tables
3. **Advanced**: Learn about Z-ORDER, data skipping, vacuum retention
4. **Real Data**: Apply these concepts to actual datasets in your workspace
5. **Performance**: Study Spark execution plans and optimization techniques

---

## Congratulations! 🎉

You've completed a comprehensive PySpark module covering:
* ✓ Delta Lake operations (hands-on)
* ✓ Spark SQL queries (hands-on)
* ✓ Temporary views (hands-on)
* ✓ CSV concepts (theory)

**All with pure PySpark - no pandas workarounds!**

You're now equipped to:
* Build production data pipelines with Delta Lake
* Write complex SQL queries in Spark
* Work seamlessly between SQL and DataFrame APIs
* Ace PySpark technical interviews